# 4교시: 동적 페이지 크롤링 — 아이차트 레코드(퍼펙트 올킬)

`t03_record_playwright.py`를 수업용으로 단계별로 나눈 노트북입니다. 위에서부터 셀을 하나씩 실행하세요.

| 단계 | 하는 일 |
|---|---|
| ① | requests로 받아 보기 → 곡이 **0개**인 것 확인 |
| ② | Playwright로 진짜 브라우저를 열어 보기 |
| ③ | 한 번 읽어 보기 → 일부만 보이는 이유(가상 스크롤) |
| ④ | 스크롤하면서 모두 모으기 |
| ⑤ | 엑셀로 저장 |

**준비 (처음 한 번, 터미널에서)**
```
pip install -r requirements.txt
playwright install chromium
```

In [12]:
URL = "https://www.ichart.kr/record/ichart?recordType=perfect-all-kill"  # 수집할 주소

## ① requests로 받아 보기

3교시처럼 requests로 받아서 곡 링크를 세어 봅니다.

In [13]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}  # 브라우저인 척
res = requests.get(URL, headers=headers, timeout=10)
res.encoding = "utf-8"
soup = BeautifulSoup(res.text, "html.parser")

print("상태 코드:", res.status_code)
print("곡 링크 수:", len(soup.select('a[href*="/song/"]')))  # 곡 하나 = /song/번호 링크

상태 코드: 200
곡 링크 수: 0


상태 코드는 200(성공)인데 곡은 **0개**입니다.

- 브라우저 **Ctrl+U**(페이지 소스): 서버가 처음 보내 준 HTML → 곡 목록이 없음
- 브라우저 **F12 → Elements**: 자바스크립트가 실행된 뒤의 화면 → 곡 목록이 있음

requests는 자바스크립트를 실행하지 못합니다. 그래서 **진짜 브라우저를 조종하는 Playwright**를 씁니다.

## ② Playwright로 브라우저 열기

> 노트북에서는 `async_playwright`와 `await`를 씁니다.
> (`.py` 파일에서 쓰는 `sync_playwright`는 노트북 안에서 오류가 납니다.)

아래 셀을 실행하면 크롬 창이 뜹니다. 창을 닫지 말고 다음 셀로 넘어가세요.

In [14]:
from playwright.async_api import async_playwright

pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=False)  # False = 크롬 창을 띄워서 보여 줌
page = await browser.new_page()
await page.goto(URL)
await page.wait_for_selector('a[href*="/song/"]', timeout=15000)  # 곡 링크가 나타날 때까지 (최대 15초)

print("열린 주소:", page.url)

열린 주소: https://www.ichart.kr/record/ichart?recordType=perfect-all-kill


## ③ 한 번 읽어 보기

`page.content()`는 **지금 브라우저에 그려진 HTML**을 돌려줍니다.
여기서부터는 3교시와 똑같이 BeautifulSoup으로 읽습니다.

F12로 본 목록 한 줄의 구조:

```html
<a href="/song/36882">
  <article>
    <span>01</span>                          ← 순위
    <img alt="Golden">                       ← 이미지 alt에는 곡명만 있음
    <p>ejae, KPop Demon Hunters Cast …</p>   ← 가수
    <h3>Golden</h3>                          ← 곡명
    <time>발매일 : 2025. 06. 20</time>       ← 발매일
    <div><span>1484</span><span>회</span></div>  ← 퍼펙트 올킬 횟수 (마지막 div)
  </article>
</a>
```

In [15]:
import pandas as pd

# 지금 브라우저에 그려진 HTML 읽기
soup = BeautifulSoup(await page.content(), "html.parser")

rows = []
for a in soup.select('a[href*="/song/"]'):  # 곡 하나 = /song/번호 링크
    if a.select_one("h3") is None:  # 아직 덜 그려진 줄은 건너뜀
        continue

    rank = a.select_one("span").get_text(strip=True)  # 첫 span = 순위 ("01")
    title = a.select_one("h3").get_text(strip=True)
    artist = a.select_one("p").get_text(strip=True)
    # "발매일 : 2025. 06. 20" → ":" 뒤만 → "2025. 06. 20"
    released = a.select_one("time").get_text(strip=True).split(":")[-1].strip()
    count = a.select_one("article > div:last-child span").get_text(strip=True)  # 마지막 div = 횟수

    rows.append({
        "순위": int(rank),  # "01" → 1
        "곡명": title,
        "가수": artist,
        "발매일": released,
        "횟수": count,
    })

print("한 번에 읽은 곡 수:", len(rows))
pd.DataFrame(rows)

한 번에 읽은 곡 수: 16


,순위,곡명,가수,발매일,횟수
0,1,Golden,"ejae, KPop Demon Hunters Cast, HUNTR/X, AUDREY...",2025. 06. 20,1484
1,2,Good Goodbye,화사(HWASA),2025. 10. 15,750
2,3,Ditto,NewJeans,2023. 01. 02,655
3,4,Dynamite,방탄소년단,2020. 08. 21,610
4,5,Celebrity,아이유(IU),2021. 03. 25,462
5,6,I AM,IVE(아이브),2023. 04. 10,359
6,7,Love wins all,아이유(IU),2024. 02. 20,339
7,8,아무노래,지코 (ZICO),2020. 01. 13,330
8,9,Supernova,aespa,2024. 05. 27,317
9,10,APT.,"로제 (ROSÉ), Bruno Mars",2024. 10. 18,282


목록은 100곡이 넘는데 **10여 곡만** 읽혔습니다.

이 페이지는 **가상 스크롤** 목록입니다. 화면 근처의 줄만 HTML에 그려 두고,
스크롤하면 위쪽 줄은 지우고 아래쪽 줄을 새로 그립니다.
(F12 → Elements를 열어 둔 채 브라우저 창을 스크롤해 보면 줄이 바뀌는 게 보입니다.)

한 번만 스크롤해서 확인해 봅니다.

In [16]:
# 스크롤 전: 지금 HTML에 있는 순위 모으기
soup = BeautifulSoup(await page.content(), "html.parser")
ranks = []
for a in soup.select('a[href*="/song/"]'):
    if a.select_one("h3") is not None:
        ranks.append(int(a.select_one("span").get_text(strip=True)))
print("스크롤 전:", ranks[0], "위 ~", ranks[-1], "위")

await page.mouse.wheel(0, 3000)    # 아래로 3000px
await page.wait_for_timeout(1000)  # 새 줄이 그려질 시간

# 스크롤 후: 같은 방법으로 다시 모으기
soup = BeautifulSoup(await page.content(), "html.parser")
ranks = []
for a in soup.select('a[href*="/song/"]'):
    if a.select_one("h3") is not None:
        ranks.append(int(a.select_one("span").get_text(strip=True)))
print("스크롤 후:", ranks[0], "위 ~", ranks[-1], "위")

스크롤 전: 1위 ~ 16위 (16줄)
스크롤 후: 25위 ~ 53위 (29줄)


스크롤 후에는 앞쪽 순위가 HTML에서 **사라졌습니다**.
그래서 끝까지 내린 뒤 한 번에 읽으면 안 되고, **조금씩 내리면서 그때그때 읽어 모아야** 합니다.

## ④ 스크롤하면서 모두 모으기

- 맨 위로 돌아가서 600px씩 내리며 매번 읽습니다.
- 딕셔너리 키를 링크 주소로 써서, 같은 곡을 여러 번 읽어도 한 번만 남깁니다.
- 새 곡이 3번 연속 안 나오면 맨 아래까지 온 것으로 보고 멈춥니다.

### 반복문 한 바퀴에서 하는 일

책을 한 쪽씩 넘기면서 처음 보는 곡만 공책에 옮겨 적는 것과 같습니다.

| 순서 | 코드 | 뜻 |
|---|---|---|
| 1 | `before = len(songs)` | 읽기 **전** 모은 곡 수를 적어 둠 |
| 2 | `songs[a["href"]] = {...}` | 지금 화면의 곡을 읽어 공책(`songs`)에 적음 |
| 3 | `if len(songs) == before:` | 곡 수가 그대로면 "새 곡 없음" 횟수 +1, 늘었으면 0으로 |
| 4 | `if no_new >= 3: break` | 3번 연속 새 곡이 없으면 맨 아래 → 반복 끝 |
| 5 | `page.mouse.wheel(0, 600)` | 마우스 휠로 600px 아래로 |
| 6 | `page.wait_for_timeout(500)` | 0.5초 기다림 (새 줄이 그려질 시간) |

**`for _ in range(100):`의 `_`는 무엇인가요?**
`for i in range(100):`이라고 쓰면 `i`에 0, 1, 2 … 99가 차례로 들어갑니다.
여기서는 "몇 번째인지"는 쓸 일이 없고 **100번 반복**만 필요합니다.
그래서 "이 값은 안 쓴다"는 뜻으로 이름 대신 `_`를 씁니다. `i`로 바꿔도 동작은 똑같습니다.

```python
for i in range(3):
    print(i)        # 0, 1, 2  ← 번호를 쓸 때는 i

for _ in range(3):
    print("안녕")   # 안녕 ×3  ← 번호가 필요 없을 때는 _
```

**왜 같은 곡이 두 번 들어가지 않나요?**
`songs`는 딕셔너리이고 키가 곡 링크(`/song/36882`)입니다.
딕셔너리는 이미 있는 키에 넣으면 **덮어쓰고**, 없는 키만 **새로 추가**합니다.
그래서 같은 곡을 여러 번 읽어도 한 번만 남고, `len(songs)`는 처음 보는 곡이 있을 때만 늘어납니다.

```python
d = {"/song/1": "Golden"}
d["/song/1"] = "Golden"   # 이미 있는 키 → 덮어쓰기
d["/song/2"] = "Ditto"    # 새 키 → 추가
print(len(d))             # 2
```

**왜 바로 멈추지 않고 3번이나 기다리나요?**
화면이 늦게 그려지면 한 번쯤은 새 곡이 안 보일 수 있습니다. 한 번 만에 멈추면 아직 남은 곡을 놓칠 수 있어서 3번 연속일 때만 끝으로 봅니다.

**왜 최대 100번인가요?**
혹시 끝을 알아채지 못해도 100번 내리면 무조건 멈추게 하는 안전장치입니다(무한 반복 방지).

In [17]:
await page.evaluate("window.scrollTo(0, 0)")  # 맨 위로 돌아가서 시작
await page.wait_for_timeout(1000)

songs = {}  # 모은 곡 {링크: 곡 정보} — 링크가 키라서 같은 곡은 한 번만 남음
no_new = 0  # 새 곡이 "연속으로" 안 나온 횟수

# _ = 이름 없는 변수. 반복 번호(0, 1, 2 …)를 쓰지 않을 때 i 대신 씀
for _ in range(100):  # 최대 100번 반복 (무한 반복 방지)
    before = len(songs)  # 1) 읽기 전 곡 수 적어 두기

    # 2) 지금 화면의 곡을 읽어 songs에 적기 (③ 셀과 같은 코드)
    soup = BeautifulSoup(await page.content(), "html.parser")
    for a in soup.select('a[href*="/song/"]'):
        if a.select_one("h3") is None:  # 아직 덜 그려진 줄은 건너뜀
            continue
        # 같은 링크면 덮어쓰고, 처음 보는 링크면 새로 추가
        songs[a["href"]] = {
            "순위": int(a.select_one("span").get_text(strip=True)),
            "곡명": a.select_one("h3").get_text(strip=True),
            "가수": a.select_one("p").get_text(strip=True),
            "발매일": a.select_one("time").get_text(strip=True).split(":")[-1].strip(),
            "횟수": a.select_one("article > div:last-child span").get_text(strip=True),
        }

    # 3) 곡 수가 그대로면 새 곡 없음 +1, 늘었으면 0으로 되돌림
    if len(songs) == before:
        no_new = no_new + 1
    else:
        no_new = 0

    # 4) 3번 연속 새 곡 없음 = 맨 아래 도착 → 반복문 빠져나가기
    if no_new >= 3:
        break

    await page.mouse.wheel(0, 600)    # 5) 마우스 휠로 아래로 600px (화면 높이보다 작게)
    await page.wait_for_timeout(500)  # 6) 새 줄이 그려질 때까지 0.5초 기다리기

print("모은 곡 수:", len(songs))

모은 곡 수: 139


다 모았으면 브라우저를 닫습니다.

In [19]:
await browser.close()
await pw.stop()

## ⑤ 표로 정리하고 엑셀로 저장

In [18]:
df = pd.DataFrame(songs.values()).sort_values("순위", ignore_index=True)
print("순위가 1부터 빠짐없이 이어지나?", list(df["순위"]) == list(range(1, len(df) + 1)))
df

순위가 1부터 빠짐없이 이어지나? True


,순위,곡명,가수,발매일,횟수
0,1,Golden,"ejae, KPop Demon Hunters Cast, HUNTR/X, AUDREY...",2025. 06. 20,1484
1,2,Good Goodbye,화사(HWASA),2025. 10. 15,750
2,3,Ditto,NewJeans,2023. 01. 02,655
3,4,Dynamite,방탄소년단,2020. 08. 21,610
4,5,Celebrity,아이유(IU),2021. 03. 25,462
...,...,...,...,...,...
134,135,가을 아침,아이유(IU),2017. 09. 22,-
135,136,썸 탈꺼야,볼빨간사춘기,2017. 09. 28,-
136,137,연애소설 (Feat. 아이유),에픽하이 (EPIK HIGH),2017. 10. 23,-
137,138,기억의 빈자리,나얼,2017. 11. 29,-


In [20]:
from pathlib import Path

path = Path.cwd() / "ichart_record.xlsx"  # 노트북이 있는 폴더
df.to_excel(path, index=False)
print("저장 완료:", path)

저장 완료: c:\Users\artis\Downloads\crawlday_code\ichart_record.xlsx


## 정리

| | requests (3교시) | Playwright (4교시) |
|---|---|---|
| 받는 것 | 서버가 처음 보낸 HTML | 자바스크립트 실행 후 화면의 HTML |
| 속도 | 빠름 | 느림 (브라우저를 띄움) |
| 쓰는 때 | Ctrl+U에 데이터가 보일 때 | F12에만 데이터가 보일 때 |

HTML을 받은 뒤 BeautifulSoup으로 읽는 부분은 둘 다 같습니다.